# Exportar Base Testes Abnormal

Gera arquivos Parquet únicos no volume `01_apuracao_demanda/out` para download: sales orders dos últimos 13 meses fechados (mês da data máxima + 12 anteriores), cadastro de materiais e forecast enriched agregado.

## Sales Orders - Últimos 13 Meses

In [0]:
from datetime import timedelta
from dateutil.relativedelta import relativedelta
from pyspark.sql import functions as F

SOURCE_TABLE = "parts_hdbk_sandbox.dt_sales_orders.raw_sales_order"
OUT_DIR = "/Volumes/parts_hdbk_sandbox/_file_orchestrator/demand/01_apuracao_demanda/out"
N_MONTHS = 13  # mês da data máxima + 12 anteriores

# Ancora no último mês com dados (a tabela é carga histórica e pode não chegar até hoje)
max_date = spark.table(SOURCE_TABLE).agg(F.max("data")).first()[0]
end_excl = (max_date.replace(day=1) + relativedelta(months=1)).date()
start_incl = (max_date.replace(day=1) - relativedelta(months=N_MONTHS - 1)).date()

print(f"Data máxima na tabela: {max_date}")
print(f"Janela: {start_incl} (inclusive) até {end_excl} (exclusive)")

In [0]:
df = (
    spark.table(SOURCE_TABLE)
    .where((F.col("data") >= F.lit(start_incl)) & (F.col("data") < F.lit(end_excl)))
)

file_name = f"raw_sales_order_{start_incl:%Y%m}_{(end_excl - timedelta(days=1)):%Y%m}.parquet"
tmp_dir = f"{OUT_DIR}/_tmp_{file_name}"
final_path = f"{OUT_DIR}/{file_name}"

# coalesce(1) gera um único part-file, que é movido para um nome amigável
df.coalesce(1).write.mode("overwrite").parquet(tmp_dir)
part_file = [f.path for f in dbutils.fs.ls(tmp_dir) if f.name.endswith(".parquet")][0]
dbutils.fs.mv(part_file, final_path)
dbutils.fs.rm(tmp_dir, recurse=True)

print(f"Arquivo gerado: {final_path}")

In [0]:
chk = spark.read.parquet(final_path)
size_mb = [f.size for f in dbutils.fs.ls(OUT_DIR) if f.name == file_name][0] / 1024**2

print(f"Linhas na tabela (janela): {df.count():,}")
print(f"Linhas no parquet:         {chk.count():,}")
print(f"Tamanho do arquivo:        {size_mb:,.1f} MB")

display(
    chk.groupBy(F.date_format("data", "yyyy-MM").alias("mes"))
    .agg(F.count("*").alias("linhas"), F.sum("quantidade").alias("quantidade"))
    .orderBy("mes")
)

## Cadastro de Materiais

Exporta colunas selecionadas de `parts_hdbk_sandbox.pr_cadastrao.material_cadastrao` para um Parquet único no mesmo volume. Regra: quando `item_principal_cadeia` for nulo, vazio, `0` ou `SIM`, assume o próprio `material`.

In [0]:
MAT_TABLE = "parts_hdbk_sandbox.pr_cadastrao.material_cadastrao"

ipc = F.trim(F.col("item_principal_cadeia"))
ipc_invalido = ipc.isNull() | (ipc == "") | ipc.rlike(r"^0+(\.0+)?$") | (F.upper(ipc) == "SIM")

df_mat = (
    spark.table(MAT_TABLE)
    .select(
        "material",
        "empresa",
        "descricao_curta_portugues",
        "descricao_curta_ingles",
        F.when(ipc_invalido, F.col("material")).otherwise(F.col("item_principal_cadeia")).alias("item_principal_cadeia"),
        "prazo_entrega_prevista_lead_time_firme",
    )
)

mat_file_name = "material_cadastrao.parquet"
mat_tmp_dir = f"{OUT_DIR}/_tmp_{mat_file_name}"
mat_final_path = f"{OUT_DIR}/{mat_file_name}"

df_mat.coalesce(1).write.mode("overwrite").parquet(mat_tmp_dir)
mat_part = [f.path for f in dbutils.fs.ls(mat_tmp_dir) if f.name.endswith(".parquet")][0]
dbutils.fs.mv(mat_part, mat_final_path)
dbutils.fs.rm(mat_tmp_dir, recurse=True)

print(f"Arquivo gerado: {mat_final_path}")

In [0]:
chk_mat = spark.read.parquet(mat_final_path)
mat_size_mb = [f.size for f in dbutils.fs.ls(OUT_DIR) if f.name == mat_file_name][0] / 1024**2

def regra_aciona(c):
    c = F.trim(F.col(c))
    return c.isNull() | (c == "") | c.rlike(r"^0+(\.0+)?$") | (F.upper(c) == "SIM")

n_src = spark.table(MAT_TABLE).count()
n_regra_src = spark.table(MAT_TABLE).where(regra_aciona("item_principal_cadeia")).count()
n_regra_out = chk_mat.where(regra_aciona("item_principal_cadeia")).count()

print(f"Linhas na tabela:   {n_src:,}")
print(f"Linhas no parquet:  {chk_mat.count():,}")
print(f"Tamanho do arquivo: {mat_size_mb:,.1f} MB")
print(f"Linhas na origem que acionam a regra (nulo/vazio/0/SIM): {n_regra_src:,}")
print(f"Restantes no parquet com nulo/vazio/0/SIM:              {n_regra_out:,}")

display(chk_mat.limit(10))

## Forecast Enriched

Exporta os **últimos 7 ciclos** (`forecast_cycle` máximo + 6 anteriores) de `parts_hdbk_sandbox.pr_forecast.refined_forecast_enriched` agregado por `forecast_cycle`, `segment`, `material`, `forecast_month`, com `forecast_qty` = soma (Base + Enrich, todos os mercados e departamentos).

Regra: grupos cuja **soma** (após o agrupamento) fica negativa são removidos do Parquet e detalhados (considerando todos os ciclos), linha a linha da origem, em `refined_forecast_enriched_negativos_investigacao.csv` (separador `;`). Linhas negativas individuais da origem (ajustes Enrich) continuam entrando normalmente na soma.

In [0]:
FC_TABLE = "parts_hdbk_sandbox.pr_forecast.refined_forecast_enriched"

FC_KEYS = ["forecast_cycle", "segment", "material", "forecast_month"]
NEG_THRESHOLD = -1e-6  # abaixo disso é negativo real; entre -1e-6 e 0 é apenas ruído de ponto flutuante
FC_N_CYCLES = 7  # ciclo máximo + 6 anteriores

# Janela de ciclos ancorada no último forecast_cycle disponível
fc_max_cycle = spark.table(FC_TABLE).agg(F.max("forecast_cycle")).first()[0]
fc_start_cycle = fc_max_cycle.replace(day=1) - relativedelta(months=FC_N_CYCLES - 1)
fc_in_window = (F.col("forecast_cycle") >= F.lit(fc_start_cycle)) & (F.col("forecast_cycle") <= F.lit(fc_max_cycle))
print(f"Ciclos exportados: {fc_start_cycle} até {fc_max_cycle}")

# Agrega TODO o histórico (o CSV de investigação de negativos cobre todos os ciclos)
df_fc_all = (
    spark.table(FC_TABLE)
    .groupBy(*FC_KEYS)
    .agg(F.sum("forecast_qty").alias("forecast_qty"))
)

# Negativo avaliado APÓS a soma
df_fc_neg_all = df_fc_all.where(F.col("forecast_qty") < NEG_THRESHOLD)          # todos os ciclos -> CSV
df_fc_neg = df_fc_neg_all.where(fc_in_window)                                     # negativos removidos da janela
df_fc = df_fc_all.where(fc_in_window & (F.col("forecast_qty") >= NEG_THRESHOLD))  # export

fc_file_name = "refined_forecast_enriched.parquet"
fc_tmp_dir = f"{OUT_DIR}/_tmp_{fc_file_name}"
fc_final_path = f"{OUT_DIR}/{fc_file_name}"

df_fc.coalesce(1).write.mode("overwrite").parquet(fc_tmp_dir)
fc_part = [f.path for f in dbutils.fs.ls(fc_tmp_dir) if f.name.endswith(".parquet")][0]
dbutils.fs.mv(fc_part, fc_final_path)
dbutils.fs.rm(fc_tmp_dir, recurse=True)

print(f"Arquivo gerado: {fc_final_path}")

In [0]:
# Para cada grupo cuja SOMA ficou negativa (calculado após o agrupamento, em TODOS os ciclos),
# traz todas as linhas de origem que compõem a soma, para entender a causa.
neg_keys = df_fc_neg_all.withColumnRenamed("forecast_qty", "forecast_qty_total_grupo")

df_neg_detail = (
    spark.table(FC_TABLE)
    .join(neg_keys, FC_KEYS, "inner")
    .select(
        *FC_KEYS,
        "forecast_qty_total_grupo",
        "lag",
        "forecast_type",
        "department",
        "market",
        "main_material",
        "forecast_classification",
        "forecast_qty",
        "_load_id",
        "_ingested_at",
    )
    .orderBy(*FC_KEYS, "forecast_type", "department", "market")
)

neg_csv_name = "refined_forecast_enriched_negativos_investigacao.csv"
neg_tmp_dir = f"{OUT_DIR}/_tmp_{neg_csv_name}"
neg_csv_path = f"{OUT_DIR}/{neg_csv_name}"

(df_neg_detail.coalesce(1).write.mode("overwrite")
    .option("header", True).option("sep", ";").option("encoding", "UTF-8")
    .csv(neg_tmp_dir))
neg_part = [f.path for f in dbutils.fs.ls(neg_tmp_dir) if f.name.endswith(".csv")][0]
dbutils.fs.mv(neg_part, neg_csv_path)
dbutils.fs.rm(neg_tmp_dir, recurse=True)

print(f"Grupos com soma negativa (todos os ciclos): {df_fc_neg_all.count():,}")
print(f"Linhas de origem no CSV:  {df_neg_detail.count():,}")
print(f"Arquivo gerado: {neg_csv_path}")

In [0]:
chk_fc = spark.read.parquet(fc_final_path)
fc_size_mb = [f.size for f in dbutils.fs.ls(OUT_DIR) if f.name == fc_file_name][0] / 1024**2

src_by_cycle = spark.table(FC_TABLE).where(fc_in_window).groupBy("forecast_cycle").agg(F.count("*").alias("linhas_origem"), F.sum("forecast_qty").alias("qty_origem"))
out_by_cycle = chk_fc.groupBy("forecast_cycle").agg(F.count("*").alias("linhas_parquet"), F.sum("forecast_qty").alias("qty_parquet"))
neg_by_cycle = df_fc_neg.groupBy("forecast_cycle").agg(F.count("*").alias("linhas_negativas_removidas"), F.sum("forecast_qty").alias("qty_negativa_removida"))

print(f"Ciclos no parquet:            {chk_fc.select('forecast_cycle').distinct().count()} "
      f"({chk_fc.agg(F.min('forecast_cycle')).first()[0]} a {chk_fc.agg(F.max('forecast_cycle')).first()[0]})")
print(f"Linhas no parquet:            {chk_fc.count():,}")
print(f"Linhas negativas removidas:   {df_fc_neg.count():,}")
print(f"Negativas restantes no arquivo: {chk_fc.where(F.col('forecast_qty') < NEG_THRESHOLD).count():,}")
print(f"Tamanho do arquivo: {fc_size_mb:,.1f} MB")

# Conferência: origem = parquet + negativas removidas
display(
    src_by_cycle.join(out_by_cycle, "forecast_cycle", "full")
    .join(neg_by_cycle, "forecast_cycle", "left")
    .fillna(0, ["linhas_negativas_removidas", "qty_negativa_removida"])
    .withColumn("diff_qty", F.round(F.col("qty_origem") - F.col("qty_parquet") - F.col("qty_negativa_removida"), 3))
    .orderBy("forecast_cycle")
)